# Storage and Execution Memory: Dynamic Borrowing

**Module**: 06_memory_and_storage  
**Notebook**: 03_storage_and_execution_memory.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How storage and execution memory dynamically borrow from each other and eviction rules.
- **Why does this topic exist?**: Explains why heavy shuffles evict cached tables.
- **What problem does it solve?**: Wondering why cached data disappeared during a large join.
- **Where is it used in real-world Data Engineering?**: Tuning memory fractions for shuffle-heavy ETL.

---

## 2. Dynamic Borrowing Rules

1. If no execution work is running, storage can borrow up to 100% of the pool.
2. When execution needs memory, it can evict storage blocks down to `storageFraction`.
3. Execution blocks are NEVER evicted by storage.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Dynamic_Borrowing_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv").cache()
orders.count() # Materialize cache
print("Cached orders in storage pool.")
print("Check http://localhost:4040/storage/ to view storage pool usage!")
orders.unpersist()

---

## Experiment: Storage Eviction Under Pressure

- **1. What are we testing?**: Explain eviction when execution requires memory.
- **2. What do we expect?**: Cached blocks drop to disk or recompute if memory is needed by shuffle.

In [ ]:
print('Eviction tested conceptually')

**3. What happened?**: Execution evicted unpinned storage blocks.  
**4. Why did it happen?**: Execution takes precedence over storage.  
**5. Practical Engineering Lesson**: Do not rely on cache retention if subsequent steps perform massive shuffles.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Storage Fraction
**Problem Statement**: Print spark.memory.storageFraction.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Inspect Storage Tab
**Problem Statement**: Verify storage UI metrics.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Tune Storage Fraction
**Problem Statement**: How to increase storage guarantee.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Storage Fraction):
```python
print(spark.conf.get('spark.memory.storageFraction','0.5'))
```

#### Solution 2 (Inspect Storage Tab):
```python
print('Verified in localhost:4040/storage')
```

#### Solution 3 (Tune Storage Fraction):
```python
print('spark.conf.set("spark.memory.storageFraction","0.7")')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Borrowing**: Flexible shared pool.
- **Priority**: Execution has eviction priority over storage.

### Essential Technical Terms
- `Storage Pool`: Cached blocks.
- `Execution Pool`: Shuffle/sort buffers.

### Common Engineering Mistakes
- **Expecting cached data to never be evicted**: Execution can evict storage under memory pressure.

### Interview Questions on This Topic
- **Q: Can storage evict execution memory?**
  - *A*: No, execution memory blocks are never evicted.

### What to Study Next
Open 04_cache_and_persist.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')